In [1]:
from safari_tools import SaFariAnnotation
from safari_sampler import SaFariSampler
from safari_downloader import SaFariDownloader
from safari_vww import SaFariVWW
from pathlib import Path
import pandas as pd
import json

In [ ]:
def download_sample(annotations_path, plan_tag_name, split: str):
    ann = SaFariAnnotation.from_json(annotations_path)   # load the object

    s = SaFariSampler(ann, pct=0.10, seed=0)
    s.save(out_dir="../data/plans", tag=plan_tag_name)

    # save writes plan_<tag>.json — load the matching path
    plan = json.load(open(f"../data/plans/plan_{plan_tag_name}.json"))

    dl = SaFariDownloader(ann, split, "../data/google_sa_fari")
    dl.download_videos(plan)

download_sample("../data/hf_sa_fari/annotation/sa_fari_train.json",
                "train_download_plan", "train")

download_sample("../data/hf_sa_fari/annotation/sa_fari_test.json",
                "test_download_plan", "test")

plan:     ../data/plans/plan_train_download_plan.json  (10776 videos)
manifest: ../data/plans/manifest_train_download_plan.csv  (98506 frames)
meta:     ../data/plans/meta_train_download_plan.json
  present=71157  empty=27349
sa_fari_000834: 0 downloaded, 9 skipped, 0 failed
sa_fari_000836: 0 downloaded, 13 skipped, 0 failed
sa_fari_000837: 0 downloaded, 10 skipped, 0 failed
sa_fari_000838: 0 downloaded, 2 skipped, 0 failed
sa_fari_000839: 0 downloaded, 10 skipped, 0 failed
sa_fari_000840: 0 downloaded, 25 skipped, 0 failed
sa_fari_000841: 0 downloaded, 9 skipped, 0 failed
sa_fari_000842: 0 downloaded, 9 skipped, 0 failed
sa_fari_000843: 0 downloaded, 7 skipped, 0 failed
sa_fari_000844: 0 downloaded, 8 skipped, 0 failed
sa_fari_000845: 0 downloaded, 10 skipped, 0 failed
sa_fari_000846: 0 downloaded, 9 skipped, 0 failed
sa_fari_000847: 0 downloaded, 10 skipped, 0 failed
sa_fari_000848: 0 downloaded, 10 skipped, 0 failed
sa_fari_000849: 0 downloaded, 7 skipped, 0 failed
sa_fari_000850: 0

In [ ]:
ann   = SaFariAnnotation.from_json("../data/hf_sa_fari/annotation/sa_fari_train.json")
ann_t = SaFariAnnotation.from_json("../data/hf_sa_fari/annotation/sa_fari_test.json")
vww   = SaFariVWW(ann)      # 2.5%, drop
vww_t = SaFariVWW(ann_t)

# step 3: labeling (2.5% rule, small-only frames dropped, setup videos excluded)
vww.relabel_manifest("../data/plans/manifest_train_download_plan.csv",
                     "../data/plans/manifest_vww_train.csv");
vww_t.relabel_manifest("../data/plans/manifest_test_download_plan.csv",
                       "../data/plans/manifest_vww_test.csv");

RELABEL  ../data/plans/manifest_train_download_plan.csv  ->  ../data/plans/manifest_vww_train.csv
rule:        bbox > 2.50%   policy 'drop'
source rows: 98506   excluded: 14   dropped: 29922   flipped 1->0: 0   written: 68570
result:      present 41235  empty 27335  (60.14% present)
  [PASS] manifest labels agree with annotation
  [PASS] no duplicate paths
  [PASS] every path came from the source manifest
  [PASS] rows conserved (written + excluded + dropped == source)
  [PASS] excluded videos absent
  [PASS] labels only 0/1
  [PASS] no empty frame became animal
meta:        ../data/plans/manifest_vww_train.meta.json
RELABEL  ../data/plans/manifest_test_download_plan.csv  ->  ../data/plans/manifest_vww_test.csv
rule:        bbox > 2.50%   policy 'drop'
source rows: 7861   excluded: 0   dropped: 2800   flipped 1->0: 0   written: 5061
result:      present 2401  empty 2660  (47.44% present)
  [PASS] manifest labels agree with annotation
  [PASS] no duplicate paths
  [PASS] every path came

In [5]:
vww.split_manifest("../data/plans/manifest_vww_train.csv",
                   out_dir="../data/plans", val_frac=0.15, seed=0);

SPLIT  ../data/plans/manifest_vww_train.csv
seed / val target:  0 / 0.15  (actual 0.153)
videos:             9626  (with location 9559, without 67)
groups:             605 locations + 67 video-fallback
largest group:      3234 frames
train:   8258 videos    58082 frames  present  34738 (59.81%)  empty  23344
val  :   1368 videos    10488 frames  present   6497 (61.95%)  empty   3991
  [PASS] no video in both splits
  [PASS] no location group in both splits
  [PASS] excluded videos absent
  [PASS] no duplicate paths
  [PASS] labels only 0/1
  [PASS] rows conserved (train + val == manifest)
  [PASS] every split path came from the manifest
wrote: split_train.csv, split_val.csv, split_meta.json -> ../data/plans


In [ ]:
ROOT  = Path("..")                                     # ~/safari, seen from code/
PLANS = ROOT / "data/plans"
SRC   = ROOT / "data/google_sa_fari/sa_fari"

for size in (96, 128):
    CACHE = ROOT / f"data/vww{size}"
    SaFariVWW.build_cache(
        [PLANS / "split_train.csv", PLANS / "split_val.csv"],
        src_root=SRC / "sa_fari_train/JPEGImages_6fps",
        cache_root=CACHE / "sa_fari_train",
        image_size=size, workers=8,
    )
    SaFariVWW.build_cache(
        PLANS / "manifest_vww_test.csv",
        src_root=SRC / "sa_fari_test/JPEGImages_6fps",
        cache_root=CACHE / "sa_fari_test",
        image_size=size, workers=8,
    )

  5000/68570  written 0  skipped 5000  errors 0
  10000/68570  written 0  skipped 10000  errors 0
  15000/68570  written 0  skipped 15000  errors 0
  20000/68570  written 0  skipped 20000  errors 0
  25000/68570  written 0  skipped 25000  errors 0
  30000/68570  written 0  skipped 30000  errors 0
  35000/68570  written 0  skipped 35000  errors 0
  40000/68570  written 0  skipped 40000  errors 0
  45000/68570  written 0  skipped 45000  errors 0
  50000/68570  written 0  skipped 50000  errors 0
  55000/68570  written 0  skipped 55000  errors 0
  60000/68570  written 0  skipped 60000  errors 0
  65000/68570  written 0  skipped 65000  errors 0
CACHE  68570 frames 96x96 -> ../data/vww96/sa_fari_train
written 0   skipped 68570   errors 0   invalid 0
  5000/5061  written 0  skipped 5000  errors 0
CACHE  5061 frames 96x96 -> ../data/vww96/sa_fari_test
written 0   skipped 5061   errors 0   invalid 0
  5000/68570  written 0  skipped 5000  errors 0
  10000/68570  written 0  skipped 10000  errors 

In [2]:
!git -C ~/safari/code/third_party/mlperf_tiny rev-parse HEAD  

4addd0fa08d216e20637637874e084895f289da4


In [3]:
!pip freeze > ~/safari/requirements_used.txt  

In [ ]:
H5="code/third_party/mlperf_tiny/benchmark/training/visual_wake_words/trained_models/vww_96.h5"
!python3 code/train_vww_safari.py --run-name scratch96_s0 --seed 0
!python3 code/train_vww_safari.py --run-name ft96_s0      --seed 0 --init $H5
!python3 code/train_vww_safari.py --run-name ft128_s0     --seed 0 --init $H5 --image-size 128

In [2]:
from pathlib import Path
src = Path("third_party/mlperf_tiny/benchmark/training/visual_wake_words/vww_model.py").read_text()
for i, line in enumerate(src.splitlines(), 1):
    if "regulariz" in line.lower() or "l2(" in line.lower():
        print(i, line.strip())
print("lines with regularization:", sum("regulariz" in l.lower() for l in src.splitlines()))

12 from tensorflow.keras.regularizers import l2
31 kernel_regularizer=l2(1e-4))(x)
42 kernel_regularizer=l2(1e-4))(x)
52 kernel_regularizer=l2(1e-4))(x)
61 kernel_regularizer=l2(1e-4))(x)
71 kernel_regularizer=l2(1e-4))(x)
80 kernel_regularizer=l2(1e-4))(x)
89 kernel_regularizer=l2(1e-4))(x)
98 kernel_regularizer=l2(1e-4))(x)
108 kernel_regularizer=l2(1e-4))(x)
117 kernel_regularizer=l2(1e-4))(x)
126 kernel_regularizer=l2(1e-4))(x)
135 kernel_regularizer=l2(1e-4))(x)
145 kernel_regularizer=l2(1e-4))(x)
155 kernel_regularizer=l2(1e-4))(x)
164 kernel_regularizer=l2(1e-4))(x)
173 kernel_regularizer=l2(1e-4))(x)
182 kernel_regularizer=l2(1e-4))(x)
191 kernel_regularizer=l2(1e-4))(x)
200 kernel_regularizer=l2(1e-4))(x)
209 kernel_regularizer=l2(1e-4))(x)
218 kernel_regularizer=l2(1e-4))(x)
227 kernel_regularizer=l2(1e-4))(x)
236 kernel_regularizer=l2(1e-4))(x)
245 kernel_regularizer=l2(1e-4))(x)
255 kernel_regularizer=l2(1e-4))(x)
264 kernel_regularizer=l2(1e-4))(x)
273 kernel_regularizer=l

In [2]:
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")      # same Keras 2 API used in training
import numpy as np, pandas as pd, tensorflow as tf
from pathlib import Path
from PIL import Image

RUNS = {"s96x96": ("scratch96_s0", 96), "96x96": ("ft96_s0", 96), "128x128": ("ft128_s0", 128)}
TR = pd.read_csv("../data/plans/split_train.csv", dtype={"label": int})
REP = TR[TR.label == 1].sample(11, random_state=0)      # as MLPerf convert_vww.py: 11 positive-class images

def img(root, p):
    return np.asarray(Image.open(Path(root) / p).convert("RGB"), np.float32) / 255.0

for name, (run, S) in RUNS.items():
    model = tf.keras.models.load_model(f"../checkpoints/{run}/last.h5", compile=False)
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    Path(f"../checkpoints/{run}/last_float32.tflite").write_bytes(conv.convert())

    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([img(f"../data/vww{S}/sa_fari_train", p)[None]] for p in REP.path)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = conv.inference_output_type = tf.int8
    Path(f"../checkpoints/{run}/last_int8.tflite").write_bytes(conv.convert())
    print("saved", name)

print("calibration frames:", list(REP.path))

I0000 00:00:1794707777.988782    1398 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794707778.417663    1398 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794707780.659561    1398 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794707782.148229    1398 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB mem

INFO:tensorflow:Assets written to: /tmp/tmptrl6q1z3/assets


INFO:tensorflow:Assets written to: /tmp/tmptrl6q1z3/assets
W0000 00:00:1794707789.578063    1398 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794707789.578126    1398 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794707789.578404    1398 reader.cc:83] Reading SavedModel from: /tmp/tmptrl6q1z3
I0000 00:00:1794707789.585467    1398 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794707789.585525    1398 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmptrl6q1z3
I0000 00:00:1794707789.621007    1398 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1794707789.626309    1398 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794707789.785930    1398 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmptrl6q1z3
I0000 00:00:1794707789.835661    1398 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Too

INFO:tensorflow:Assets written to: /tmp/tmpa4spkm_s/assets


INFO:tensorflow:Assets written to: /tmp/tmpa4spkm_s/assets
/home/kitcovid-0738/safari/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1794707795.408530    1398 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794707795.408566    1398 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794707795.408684    1398 reader.cc:83] Reading SavedModel from: /tmp/tmpa4spkm_s
I0000 00:00:1794707795.414229    1398 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794707795.414248    1398 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpa4spkm_s
I0000 00:00:1794707795.456530    1398 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794707795.602684    1398 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpa4spkm_s
I0000 00:00:1

saved s96x96
INFO:tensorflow:Assets written to: /tmp/tmptvbalknh/assets


... (5 outputs left out)

INFO:tensorflow:Assets written to: /tmp/tmpftjgy10w/assets


INFO:tensorflow:Assets written to: /tmp/tmpftjgy10w/assets


saved 128x128
calibration frames: ['sa_fari_006931/00022.jpg', 'sa_fari_002524/00050.jpg', 'sa_fari_009112/00053.jpg', 'sa_fari_005933/00011.jpg', 'sa_fari_003962/00054.jpg', 'sa_fari_009221/00000.jpg', 'sa_fari_007995/00042.jpg', 'sa_fari_009973/00012.jpg', 'sa_fari_007257/00091.jpg', 'sa_fari_008985/00060.jpg', 'sa_fari_008978/00000.jpg']


/home/kitcovid-0738/safari/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1794707820.416414    1398 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794707820.416480    1398 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794707820.416701    1398 reader.cc:83] Reading SavedModel from: /tmp/tmpftjgy10w
I0000 00:00:1794707820.425371    1398 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794707820.425446    1398 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpftjgy10w
I0000 00:00:1794707820.477675    1398 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794707820.674330    1398 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpftjgy10w
I0000 00:00:1794707820.737991    1398 loader.cc:471] SavedModel load for